In [1]:
from pathlib import Path
import os
import sys
import json
import subprocess
from datetime import datetime

direktori_aktif = Path.cwd()

if direktori_aktif.name.lower() == "notebooks":
    direktori_project = direktori_aktif.parent
else:
    direktori_project = direktori_aktif

direktori_src = direktori_project / "src"
direktori_outputs = direktori_project / "reports" / "outputs"
direktori_examples = direktori_project / "examples"
direktori_intelligence = direktori_project / "data" / "intelligence"

for folder in [direktori_src, direktori_outputs, direktori_examples, direktori_intelligence]:
    folder.mkdir(parents=True, exist_ok=True)

print("Direktori aktif notebook:", direktori_aktif)
print("Direktori project:", direktori_project)
print("Folder src:", direktori_src)
print("Folder outputs:", direktori_outputs)
print("Folder examples:", direktori_examples)
print("Folder intelligence:", direktori_intelligence)

Direktori aktif notebook: C:\Users\ASUS\PHISHING\notebooks
Direktori project: C:\Users\ASUS\PHISHING
Folder src: C:\Users\ASUS\PHISHING\src
Folder outputs: C:\Users\ASUS\PHISHING\reports\outputs
Folder examples: C:\Users\ASUS\PHISHING\examples
Folder intelligence: C:\Users\ASUS\PHISHING\data\intelligence


In [2]:
import pandas as pd

file_wajib = [
    direktori_src / "phishrisk_engine_v3.py",
    direktori_src / "url_intelligence.py",
    direktori_src / "file_static_analyzer.py",
    direktori_project / "models" / "model_terbaik_intelligence_v2.pkl",
    direktori_outputs / "daftar_fitur_intelligence_v2.json",
]

data_validasi_awal = []
for lokasi in file_wajib:
    data_validasi_awal.append({
        "nama_file": lokasi.name,
        "lokasi": str(lokasi),
        "tersedia": lokasi.exists(),
        "ukuran_kb": round(lokasi.stat().st_size / 1024, 2) if lokasi.exists() else 0,
    })

data_validasi_awal = pd.DataFrame(data_validasi_awal)
display(data_validasi_awal)

if not data_validasi_awal["tersedia"].all():
    file_hilang = data_validasi_awal.loc[~data_validasi_awal["tersedia"], "nama_file"].tolist()
    raise FileNotFoundError(f"Ada file wajib yang belum tersedia: {file_hilang}")

print("Semua file wajib tersedia.")

,nama_file,lokasi,tersedia,ukuran_kb
0,phishrisk_engine_v3.py,C:\Users\ASUS\PHISHING\src\phishrisk_engine_v3.py,True,17.66
1,url_intelligence.py,C:\Users\ASUS\PHISHING\src\url_intelligence.py,True,15.60
2,file_static_analyzer.py,C:\Users\ASUS\PHISHING\src\file_static_analyze...,True,17.95
3,model_terbaik_intelligence_v2.pkl,C:\Users\ASUS\PHISHING\models\model_terbaik_in...,True,49114.43
4,daftar_fitur_intelligence_v2.json,C:\Users\ASUS\PHISHING\reports\outputs\daftar_...,True,1.02


Semua file wajib tersedia.


In [4]:
import importlib.util
import subprocess
import sys

paket_wajib = {
    "requests": "requests",
    "dotenv": "python-dotenv",
}

paket_belum_ada = []
for nama_import, nama_pip in paket_wajib.items():
    if importlib.util.find_spec(nama_import) is None:
        paket_belum_ada.append(nama_pip)

if paket_belum_ada:
    print("Package belum ada, install sekarang:", paket_belum_ada)
    subprocess.run([sys.executable, "-m", "pip", "install", *paket_belum_ada], check=True)
else:
    print("Package sudah tersedia.")

print("Selesai cek package.")

Package sudah tersedia.
Selesai cek package.


In [5]:
lokasi_gitignore = direktori_project / ".gitignore"
lokasi_env_example = direktori_project / ".env.example"

def tambah_baris_unik(lokasi_file: Path, daftar_baris: list[str]) -> None:
    isi = lokasi_file.read_text(encoding="utf-8") if lokasi_file.exists() else ""
    baris = isi.splitlines()

    for item in daftar_baris:
        if item not in baris:
            baris.append(item)

    lokasi_file.write_text("\n".join(baris).strip() + "\n", encoding="utf-8")

tambah_baris_unik(
    lokasi_gitignore,
    [
        ".env",
        ".env.*",
        "!.env.example",
        "data/threat_cache/",
    ]
)

tambah_baris_unik(
    lokasi_env_example,
    [
        "",
        "# Optional public threat intelligence",
        "PHISHTANK_APP_KEY=optional_app_key_phishtank",
        "URLHAUS_AUTH_KEY=optional_auth_key_urlhaus",
        "PHISHRISK_PUBLIC_TI_ENABLE_PHISHTANK=1",
        "PHISHRISK_PUBLIC_TI_ENABLE_URLHAUS=1",
        "PHISHRISK_PUBLIC_TI_TIMEOUT=12",
    ]
)

print(".gitignore dan .env.example berhasil diperbarui.")
print("File .env jangan di-push ke GitHub.")

.gitignore dan .env.example berhasil diperbarui.
File .env jangan di-push ke GitHub.


In [6]:
isi_public_threat_intelligence = 'from __future__ import annotations\n\nimport os\nimport time\nfrom dataclasses import dataclass\nfrom pathlib import Path\nfrom typing import Any, Dict, Iterable, Optional\nfrom urllib.parse import urlparse\n\nimport pandas as pd\nimport requests\n\ntry:\n    from dotenv import load_dotenv\n    load_dotenv(Path(__file__).resolve().parents[1] / ".env")\nexcept Exception:\n    pass\n\n\ndef _bersihkan_url(url: Any) -> str:\n    if url is None:\n        return ""\n\n    teks = str(url).strip()\n\n    if teks and not teks.lower().startswith(("http://", "https://")):\n        teks = "https://" + teks\n\n    return teks\n\n\ndef _domain_dari_url(url: str) -> str:\n    try:\n        return urlparse(url).netloc.lower()\n    except Exception:\n        return ""\n\n\ndef _nilai_bool(data: Any) -> bool:\n    if isinstance(data, bool):\n        return data\n    if isinstance(data, str):\n        return data.strip().lower() in ["true", "1", "yes", "y"]\n    if isinstance(data, (int, float)):\n        return data == 1\n    return False\n\n\ndef _teks_pendek(nilai: Any, batas: int = 500) -> str:\n    if nilai is None:\n        return ""\n    teks = str(nilai).replace("\\x00", " ").strip()\n    return teks[:batas]\n\n\n@dataclass\nclass ThreatAPIConfig:\n    timeout: int = 12\n    jeda_request: float = 0.20\n    aktifkan_phishtank: bool = True\n    aktifkan_urlhaus: bool = True\n    phishtank_app_key: str = ""\n    urlhaus_auth_key: str = ""\n\n\nclass PhishTankClient:\n    """Client defensif untuk mengecek URL ke PhishTank."""\n\n    endpoint = "http://checkurl.phishtank.com/checkurl/"\n\n    def __init__(self, app_key: str = "", timeout: int = 12) -> None:\n        self.app_key = app_key or os.getenv("PHISHTANK_APP_KEY", "")\n        self.timeout = timeout\n\n    def cek_url(self, url: str) -> Dict[str, Any]:\n        url = _bersihkan_url(url)\n\n        hasil: Dict[str, Any] = {\n            "phishtank_checked": 1,\n            "phishtank_available": 0,\n            "phishtank_found": 0,\n            "phishtank_verified": 0,\n            "phishtank_valid": 0,\n            "phishtank_detail_url": "",\n            "phishtank_status": "belum_dicek",\n            "phishtank_error": "",\n        }\n\n        if not url:\n            hasil["phishtank_status"] = "url_kosong"\n            return hasil\n\n        payload = {"url": url, "format": "json"}\n        if self.app_key:\n            payload["app_key"] = self.app_key\n\n        headers = {"User-Agent": "phishtank/phishrisk-intelligence-system"}\n\n        try:\n            response = requests.post(\n                self.endpoint,\n                data=payload,\n                headers=headers,\n                timeout=self.timeout,\n            )\n\n            hasil["phishtank_available"] = 1\n\n            if response.status_code == 509:\n                hasil["phishtank_status"] = "rate_limit"\n                hasil["phishtank_error"] = "PhishTank rate limit."\n                return hasil\n\n            if response.status_code >= 400:\n                hasil["phishtank_status"] = f"http_{response.status_code}"\n                hasil["phishtank_error"] = _teks_pendek(response.text)\n                return hasil\n\n            data = response.json()\n            result = data.get("results", data if isinstance(data, dict) else {})\n\n            in_database = _nilai_bool(result.get("in_database"))\n            verified = _nilai_bool(result.get("verified"))\n            valid = _nilai_bool(result.get("valid"))\n\n            hasil.update({\n                "phishtank_found": int(in_database),\n                "phishtank_verified": int(verified),\n                "phishtank_valid": int(valid),\n                "phishtank_detail_url": result.get("phish_detail_page", "") or str(result.get("phish_id", "")),\n                "phishtank_status": "terdaftar" if in_database else "tidak_ditemukan",\n            })\n\n            if in_database and verified and valid:\n                hasil["phishtank_status"] = "phishing_terverifikasi"\n            elif in_database:\n                hasil["phishtank_status"] = "terdaftar_belum_valid"\n\n            return hasil\n\n        except Exception as error:\n            hasil["phishtank_status"] = "gagal"\n            hasil["phishtank_error"] = _teks_pendek(error)\n            return hasil\n\n\nclass URLhausClient:\n    """Client defensif untuk mengecek URL ke URLhaus.\n\n    URLhaus fokus pada URL malware/payload. API modern abuse.ch membutuhkan Auth-Key.\n    """\n\n    endpoint = "https://urlhaus-api.abuse.ch/v1/url/"\n\n    def __init__(self, auth_key: str = "", timeout: int = 12) -> None:\n        self.auth_key = auth_key or os.getenv("URLHAUS_AUTH_KEY", "")\n        self.timeout = timeout\n\n    def cek_url(self, url: str) -> Dict[str, Any]:\n        url = _bersihkan_url(url)\n\n        hasil: Dict[str, Any] = {\n            "urlhaus_checked": 1,\n            "urlhaus_available": 0,\n            "urlhaus_found": 0,\n            "urlhaus_query_status": "belum_dicek",\n            "urlhaus_url_status": "",\n            "urlhaus_threat": "",\n            "urlhaus_tags": "",\n            "urlhaus_reference": "",\n            "urlhaus_error": "",\n        }\n\n        if not url:\n            hasil["urlhaus_query_status"] = "url_kosong"\n            return hasil\n\n        if not self.auth_key:\n            hasil["urlhaus_query_status"] = "auth_key_belum_tersedia"\n            hasil["urlhaus_error"] = "URLhaus API membutuhkan Auth-Key gratis dari abuse.ch."\n            return hasil\n\n        headers = {\n            "Auth-Key": self.auth_key,\n            "User-Agent": "PhishRisk-Intelligence-System/1.0",\n        }\n\n        try:\n            response = requests.post(\n                self.endpoint,\n                data={"url": url},\n                headers=headers,\n                timeout=self.timeout,\n            )\n\n            hasil["urlhaus_available"] = 1\n\n            if response.status_code >= 400:\n                hasil["urlhaus_query_status"] = f"http_{response.status_code}"\n                hasil["urlhaus_error"] = _teks_pendek(response.text)\n                return hasil\n\n            data = response.json()\n            query_status = data.get("query_status", "")\n\n            hasil["urlhaus_query_status"] = query_status\n\n            if query_status == "ok":\n                hasil.update({\n                    "urlhaus_found": 1,\n                    "urlhaus_url_status": data.get("url_status", ""),\n                    "urlhaus_threat": data.get("threat", ""),\n                    "urlhaus_tags": ", ".join(data.get("tags", []) or []),\n                    "urlhaus_reference": data.get("urlhaus_reference", ""),\n                })\n            elif query_status == "no_results":\n                hasil["urlhaus_found"] = 0\n\n            return hasil\n\n        except Exception as error:\n            hasil["urlhaus_query_status"] = "gagal"\n            hasil["urlhaus_error"] = _teks_pendek(error)\n            return hasil\n\n\nclass PublicThreatIntelligence:\n    """Menggabungkan PhishTank dan URLhaus sebagai sinyal threat intelligence publik."""\n\n    def __init__(self, config: Optional[ThreatAPIConfig] = None) -> None:\n        timeout = int(os.getenv("PHISHRISK_PUBLIC_TI_TIMEOUT", "12") or "12")\n\n        if config is None:\n            config = ThreatAPIConfig(\n                timeout=timeout,\n                aktifkan_phishtank=os.getenv("PHISHRISK_PUBLIC_TI_ENABLE_PHISHTANK", "1") == "1",\n                aktifkan_urlhaus=os.getenv("PHISHRISK_PUBLIC_TI_ENABLE_URLHAUS", "1") == "1",\n                phishtank_app_key=os.getenv("PHISHTANK_APP_KEY", ""),\n                urlhaus_auth_key=os.getenv("URLHAUS_AUTH_KEY", ""),\n            )\n\n        self.config = config\n        self.phishtank = PhishTankClient(config.phishtank_app_key, config.timeout)\n        self.urlhaus = URLhausClient(config.urlhaus_auth_key, config.timeout)\n\n    def cek_url(self, url: str) -> Dict[str, Any]:\n        url = _bersihkan_url(url)\n\n        hasil: Dict[str, Any] = {\n            "url": url,\n            "domain": _domain_dari_url(url),\n            "public_ti_checked_at": time.strftime("%Y-%m-%d %H:%M:%S"),\n        }\n\n        if self.config.aktifkan_phishtank:\n            hasil.update(self.phishtank.cek_url(url))\n            time.sleep(self.config.jeda_request)\n        else:\n            hasil.update({\n                "phishtank_checked": 0,\n                "phishtank_available": 0,\n                "phishtank_found": 0,\n                "phishtank_verified": 0,\n                "phishtank_valid": 0,\n                "phishtank_detail_url": "",\n                "phishtank_status": "dinonaktifkan",\n                "phishtank_error": "",\n            })\n\n        if self.config.aktifkan_urlhaus:\n            hasil.update(self.urlhaus.cek_url(url))\n            time.sleep(self.config.jeda_request)\n        else:\n            hasil.update({\n                "urlhaus_checked": 0,\n                "urlhaus_available": 0,\n                "urlhaus_found": 0,\n                "urlhaus_query_status": "dinonaktifkan",\n                "urlhaus_url_status": "",\n                "urlhaus_threat": "",\n                "urlhaus_tags": "",\n                "urlhaus_reference": "",\n                "urlhaus_error": "",\n            })\n\n        hasil.update(self.hitung_skor_public_ti(hasil))\n        return hasil\n\n    def cek_banyak_url(self, daftar_url: Iterable[str]) -> pd.DataFrame:\n        hasil = []\n        for url in daftar_url:\n            hasil.append(self.cek_url(url))\n        return pd.DataFrame(hasil)\n\n    @staticmethod\n    def hitung_skor_public_ti(data: Dict[str, Any]) -> Dict[str, Any]:\n        skor = 0\n        sumber = []\n        alasan = []\n\n        if int(data.get("phishtank_found", 0)) == 1:\n            if int(data.get("phishtank_verified", 0)) == 1 and int(data.get("phishtank_valid", 0)) == 1:\n                skor = max(skor, 100)\n                sumber.append("PhishTank")\n                alasan.append("URL terdaftar sebagai phishing terverifikasi di PhishTank.")\n            else:\n                skor = max(skor, 75)\n                sumber.append("PhishTank")\n                alasan.append("URL ditemukan di PhishTank, tetapi status validasi perlu ditinjau.")\n\n        if int(data.get("urlhaus_found", 0)) == 1:\n            skor = max(skor, 100)\n            sumber.append("URLhaus")\n            alasan.append("URL ditemukan di URLhaus sebagai indikator malware atau payload berbahaya.")\n\n        if skor >= 90:\n            status = "terindikasi_ancaman_publik"\n            kategori = "Sangat Tinggi"\n            hasil = "Berisiko"\n            rekomendasi = "URL ditemukan pada sumber threat intelligence publik. Jangan dibuka, jangan login, dan lakukan pengecekan manual."\n        elif skor >= 70:\n            status = "perlu_tinjauan_threat_intelligence"\n            kategori = "Tinggi"\n            hasil = "Perlu Tinjauan"\n            rekomendasi = "URL ditemukan pada sumber eksternal, tetapi statusnya perlu dibaca ulang sebelum mengambil keputusan."\n        else:\n            status = "tidak_ditemukan_di_public_ti"\n            kategori = "Rendah"\n            hasil = "Tidak Ada Temuan"\n            rekomendasi = "Tidak ada temuan dari sumber threat intelligence publik yang aktif. Tetap gunakan hasil engine utama sebagai acuan."\n\n        return {\n            "public_ti_score": skor,\n            "public_ti_status": status,\n            "public_ti_category": kategori,\n            "public_ti_result": hasil,\n            "public_ti_sources": ", ".join(sorted(set(sumber))),\n            "public_ti_reason": " ".join(alasan) if alasan else "Tidak ada temuan pada sumber threat intelligence publik yang aktif.",\n            "public_ti_recommendation": rekomendasi,\n        }\n\n    @staticmethod\n    def gabungkan_dengan_hasil_engine(hasil_engine: Dict[str, Any], hasil_public_ti: Dict[str, Any]) -> Dict[str, Any]:\n        hasil = dict(hasil_engine)\n        hasil.update(hasil_public_ti)\n\n        skor_engine = float(hasil_engine.get("skor_final", 0) or 0)\n        skor_public = float(hasil_public_ti.get("public_ti_score", 0) or 0)\n        skor_final_v4 = max(skor_engine, skor_public)\n\n        hasil_akhir_engine = hasil_engine.get("hasil_akhir", "")\n        kategori_engine = hasil_engine.get("kategori_risiko", "")\n\n        if skor_public >= 90:\n            hasil_akhir_v4 = "Berisiko"\n            kategori_v4 = "Sangat Tinggi"\n            rekomendasi_v4 = hasil_public_ti.get("public_ti_recommendation", "")\n        elif skor_public >= 70 and hasil_akhir_engine == "Terlihat Aman":\n            hasil_akhir_v4 = "Perlu Tinjauan"\n            kategori_v4 = "Tinggi"\n            rekomendasi_v4 = hasil_public_ti.get("public_ti_recommendation", "")\n        else:\n            hasil_akhir_v4 = hasil_akhir_engine\n            kategori_v4 = kategori_engine\n            rekomendasi_v4 = hasil_engine.get("rekomendasi", "")\n\n        hasil.update({\n            "skor_final_v4": round(skor_final_v4, 2),\n            "kategori_risiko_v4": kategori_v4,\n            "hasil_akhir_v4": hasil_akhir_v4,\n            "rekomendasi_v4": rekomendasi_v4,\n        })\n\n        return hasil\n'

lokasi_public_ti = direktori_src / "public_threat_intelligence.py"
lokasi_public_ti.write_text(isi_public_threat_intelligence, encoding="utf-8")

print("Modul Public Threat Intelligence berhasil dibuat:")
print(lokasi_public_ti)

Modul Public Threat Intelligence berhasil dibuat:
C:\Users\ASUS\PHISHING\src\public_threat_intelligence.py


In [7]:
isi_engine_v4 = 'from __future__ import annotations\n\nfrom pathlib import Path\nfrom typing import Any, Dict, Iterable, Tuple\n\nimport pandas as pd\n\ntry:\n    from dotenv import load_dotenv\n    load_dotenv(Path(__file__).resolve().parents[1] / ".env")\nexcept Exception:\n    pass\n\nfrom phishrisk_engine_v3 import PhishRiskEngineV3\nfrom public_threat_intelligence import PublicThreatIntelligence\n\n\nclass PhishRiskEngineV4:\n    """Engine V4: Engine V3 + Public Threat Intelligence."""\n\n    def __init__(self, direktori_project: str | Path) -> None:\n        self.direktori_project = Path(direktori_project)\n        self.engine_v3 = PhishRiskEngineV3(self.direktori_project)\n        self.public_ti = PublicThreatIntelligence()\n\n    def analisis_url(self, url: str) -> Dict[str, Any]:\n        hasil_v3 = self.engine_v3.analisis_url(url)\n        hasil_public = self.public_ti.cek_url(url)\n        return self.public_ti.gabungkan_dengan_hasil_engine(hasil_v3, hasil_public)\n\n    def analisis_banyak_url(self, daftar_url: Iterable[str]) -> pd.DataFrame:\n        hasil = []\n        for url in daftar_url:\n            hasil.append(self.analisis_url(url))\n        return pd.DataFrame(hasil)\n\n    def analisis_file(self, lokasi_file: str | Path) -> Tuple[Dict[str, Any], pd.DataFrame]:\n        hasil_file_v3, data_url_dalam_file_v3 = self.engine_v3.analisis_file(lokasi_file)\n\n        hasil_file_v4 = dict(hasil_file_v3)\n        data_url_dalam_file_v4 = data_url_dalam_file_v3.copy() if isinstance(data_url_dalam_file_v3, pd.DataFrame) else pd.DataFrame()\n\n        if not data_url_dalam_file_v4.empty and "url" in data_url_dalam_file_v4.columns:\n            hasil_public = []\n\n            for url in data_url_dalam_file_v4["url"].dropna().astype(str).tolist():\n                hasil_public.append(self.public_ti.cek_url(url))\n\n            data_public = pd.DataFrame(hasil_public)\n\n            if not data_public.empty:\n                kolom_gabung = [\n                    "url",\n                    "public_ti_score",\n                    "public_ti_status",\n                    "public_ti_result",\n                    "public_ti_sources",\n                    "public_ti_reason",\n                    "phishtank_status",\n                    "urlhaus_query_status",\n                ]\n\n                kolom_gabung = [kolom for kolom in kolom_gabung if kolom in data_public.columns]\n\n                data_url_dalam_file_v4 = data_url_dalam_file_v4.merge(\n                    data_public[kolom_gabung],\n                    on="url",\n                    how="left",\n                )\n\n                skor_public_maks = float(data_public["public_ti_score"].max()) if "public_ti_score" in data_public.columns else 0\n                jumlah_temuan_public = int((data_public.get("public_ti_score", pd.Series(dtype=float)) >= 70).sum())\n\n                skor_file_v3 = float(hasil_file_v3.get("skor_final_file_v3", hasil_file_v3.get("skor_risiko_file", 0)) or 0)\n                skor_file_v4 = max(skor_file_v3, 95 if jumlah_temuan_public > 0 else skor_file_v3)\n\n                hasil_file_v4["jumlah_url_terdeteksi_public_ti"] = jumlah_temuan_public\n                hasil_file_v4["skor_public_ti_maks_file"] = skor_public_maks\n                hasil_file_v4["skor_final_file_v4"] = round(skor_file_v4, 2)\n\n                if jumlah_temuan_public > 0:\n                    hasil_file_v4["kategori_final_file_v4"] = "Sangat Tinggi"\n                    hasil_file_v4["hasil_akhir_file_v4"] = "Berisiko"\n                    hasil_file_v4["rekomendasi_final_file_v4"] = "File mengandung URL yang ditemukan pada threat intelligence publik. Jangan dibuka langsung."\n                else:\n                    hasil_file_v4["kategori_final_file_v4"] = hasil_file_v3.get("kategori_final_file_v3", hasil_file_v3.get("kategori_risiko_file", ""))\n                    hasil_file_v4["hasil_akhir_file_v4"] = hasil_file_v3.get("hasil_akhir_file_v3", hasil_file_v3.get("hasil_akhir_file", ""))\n                    hasil_file_v4["rekomendasi_final_file_v4"] = hasil_file_v3.get("rekomendasi_final_file_v3", hasil_file_v3.get("rekomendasi_file", ""))\n\n        if "skor_final_file_v4" not in hasil_file_v4:\n            hasil_file_v4["jumlah_url_terdeteksi_public_ti"] = 0\n            hasil_file_v4["skor_public_ti_maks_file"] = 0\n            hasil_file_v4["skor_final_file_v4"] = hasil_file_v3.get("skor_final_file_v3", hasil_file_v3.get("skor_risiko_file", 0))\n            hasil_file_v4["kategori_final_file_v4"] = hasil_file_v3.get("kategori_final_file_v3", hasil_file_v3.get("kategori_risiko_file", ""))\n            hasil_file_v4["hasil_akhir_file_v4"] = hasil_file_v3.get("hasil_akhir_file_v3", hasil_file_v3.get("hasil_akhir_file", ""))\n            hasil_file_v4["rekomendasi_final_file_v4"] = hasil_file_v3.get("rekomendasi_final_file_v3", hasil_file_v3.get("rekomendasi_file", ""))\n\n        return hasil_file_v4, data_url_dalam_file_v4\n'

lokasi_engine_v4 = direktori_src / "phishrisk_engine_v4.py"
lokasi_engine_v4.write_text(isi_engine_v4, encoding="utf-8")

print("Engine V4 berhasil dibuat:")
print(lokasi_engine_v4)

Engine V4 berhasil dibuat:
C:\Users\ASUS\PHISHING\src\phishrisk_engine_v4.py


In [8]:
isi_cli_v4 = 'from __future__ import annotations\n\nimport argparse\nfrom pathlib import Path\n\nimport pandas as pd\n\ntry:\n    from dotenv import load_dotenv\n    load_dotenv(Path(__file__).resolve().parents[1] / ".env")\nexcept Exception:\n    pass\n\nfrom phishrisk_engine_v4 import PhishRiskEngineV4\n\n\ndef cari_direktori_project() -> Path:\n    return Path(__file__).resolve().parents[1]\n\n\ndef main() -> None:\n    parser = argparse.ArgumentParser(description="PhishRisk Engine V4 - Public Threat Intelligence")\n    parser.add_argument("--mode", choices=["url", "urls", "file"], default="url")\n    parser.add_argument("--input", required=True)\n    parser.add_argument("--url-column", default="url")\n    parser.add_argument("--output", default="")\n    args = parser.parse_args()\n\n    direktori_project = cari_direktori_project()\n    engine = PhishRiskEngineV4(direktori_project)\n\n    if args.mode == "url":\n        hasil = engine.analisis_url(args.input)\n        data = pd.DataFrame([hasil])\n\n        output = Path(args.output) if args.output else direktori_project / "reports" / "outputs" / "hasil_cli_engine_v4_url.csv"\n        output.parent.mkdir(parents=True, exist_ok=True)\n        data.to_csv(output, index=False)\n\n        print("Hasil Engine V4")\n        print("=" * 60)\n        print("URL:", hasil.get("url"))\n        print("Hasil V4:", hasil.get("hasil_akhir_v4"))\n        print("Kategori V4:", hasil.get("kategori_risiko_v4"))\n        print("Skor V4:", hasil.get("skor_final_v4"))\n        print("Public TI:", hasil.get("public_ti_status"))\n        print("Output:", output)\n\n    elif args.mode == "urls":\n        data_input = pd.read_csv(args.input)\n\n        if args.url_column not in data_input.columns:\n            raise ValueError(f"Kolom URL tidak ditemukan: {args.url_column}")\n\n        daftar_url = data_input[args.url_column].dropna().astype(str).tolist()\n        data = engine.analisis_banyak_url(daftar_url)\n\n        output = Path(args.output) if args.output else direktori_project / "reports" / "outputs" / "hasil_cli_engine_v4_banyak_url.csv"\n        output.parent.mkdir(parents=True, exist_ok=True)\n        data.to_csv(output, index=False)\n\n        print("Ringkasan Engine V4")\n        print("=" * 60)\n        print(data["hasil_akhir_v4"].value_counts().to_string())\n        print("Output:", output)\n\n    else:\n        hasil_file, data_url = engine.analisis_file(args.input)\n\n        output = Path(args.output) if args.output else direktori_project / "reports" / "outputs" / "hasil_cli_engine_v4_file.csv"\n        output_url = output.with_name(output.stem + "_url_dalam_file.csv")\n\n        output.parent.mkdir(parents=True, exist_ok=True)\n        pd.DataFrame([hasil_file]).to_csv(output, index=False)\n\n        if isinstance(data_url, pd.DataFrame) and not data_url.empty:\n            data_url.to_csv(output_url, index=False)\n\n        print("Hasil File Engine V4")\n        print("=" * 60)\n        print("File:", hasil_file.get("nama_file"))\n        print("Hasil:", hasil_file.get("hasil_akhir_file_v4"))\n        print("Kategori:", hasil_file.get("kategori_final_file_v4"))\n        print("Skor:", hasil_file.get("skor_final_file_v4"))\n        print("Output:", output)\n\n\nif __name__ == "__main__":\n    main()\n'

lokasi_cli_v4 = direktori_src / "run_phishrisk_v4.py"
lokasi_cli_v4.write_text(isi_cli_v4, encoding="utf-8")

print("CLI Engine V4 berhasil dibuat:")
print(lokasi_cli_v4)

CLI Engine V4 berhasil dibuat:
C:\Users\ASUS\PHISHING\src\run_phishrisk_v4.py


In [9]:
data_url_step13 = pd.DataFrame({
    "url": [
        "https://praktikum.gunadarma.ac.id",
        "https://www.bca.co.id",
        "http://bca-login-update.test",
        "http://rricrosoft.com",
        "http://rnicrosoft.com",
        "http://paypal-verify-account.test",
        "http://155.94.163.206/ai/?authenticated=true&account=login",
        "https://www.google.com",
    ]
})

lokasi_input_step13 = direktori_examples / "input_url_step13_public_ti.csv"
data_url_step13.to_csv(lokasi_input_step13, index=False)

print("Contoh input STEP 13 berhasil dibuat:")
print(lokasi_input_step13)
display(data_url_step13)

Contoh input STEP 13 berhasil dibuat:
C:\Users\ASUS\PHISHING\examples\input_url_step13_public_ti.csv


,url
0,https://praktikum.gunadarma.ac.id
1,https://www.bca.co.id
2,http://bca-login-update.test
3,http://rricrosoft.com
4,http://rnicrosoft.com
5,http://paypal-verify-account.test
6,http://155.94.163.206/ai/?authenticated=true&a...
7,https://www.google.com


In [10]:
import sys
import importlib

if str(direktori_src) not in sys.path:
    sys.path.insert(0, str(direktori_src))

import public_threat_intelligence

importlib.reload(public_threat_intelligence)

public_ti = public_threat_intelligence.PublicThreatIntelligence()

daftar_uji = data_url_step13["url"].tolist()
hasil_public_ti = public_ti.cek_banyak_url(daftar_uji)

lokasi_hasil_public_ti = direktori_outputs / "hasil_uji_public_threat_intelligence_step13.csv"
hasil_public_ti.to_csv(lokasi_hasil_public_ti, index=False)

print("Hasil uji Public Threat Intelligence disimpan:")
print(lokasi_hasil_public_ti)

kolom_tampil = [
    "url",
    "domain",
    "public_ti_score",
    "public_ti_status",
    "public_ti_result",
    "public_ti_sources",
    "phishtank_status",
    "urlhaus_query_status",
    "public_ti_reason",
    "public_ti_recommendation",
]
kolom_tampil = [kolom for kolom in kolom_tampil if kolom in hasil_public_ti.columns]

display(hasil_public_ti[kolom_tampil])

Hasil uji Public Threat Intelligence disimpan:
C:\Users\ASUS\PHISHING\reports\outputs\hasil_uji_public_threat_intelligence_step13.csv


,url,domain,public_ti_score,public_ti_status,public_ti_result,public_ti_sources,phishtank_status,urlhaus_query_status,public_ti_reason,public_ti_recommendation
0,https://praktikum.gunadarma.ac.id,praktikum.gunadarma.ac.id,0,tidak_ditemukan_di_public_ti,Tidak Ada Temuan,,gagal,auth_key_belum_tersedia,Tidak ada temuan pada sumber threat intelligen...,Tidak ada temuan dari sumber threat intelligen...
1,https://www.bca.co.id,www.bca.co.id,0,tidak_ditemukan_di_public_ti,Tidak Ada Temuan,,gagal,auth_key_belum_tersedia,Tidak ada temuan pada sumber threat intelligen...,Tidak ada temuan dari sumber threat intelligen...
2,http://bca-login-update.test,bca-login-update.test,0,tidak_ditemukan_di_public_ti,Tidak Ada Temuan,,gagal,auth_key_belum_tersedia,Tidak ada temuan pada sumber threat intelligen...,Tidak ada temuan dari sumber threat intelligen...
3,http://rricrosoft.com,rricrosoft.com,0,tidak_ditemukan_di_public_ti,Tidak Ada Temuan,,gagal,auth_key_belum_tersedia,Tidak ada temuan pada sumber threat intelligen...,Tidak ada temuan dari sumber threat intelligen...
4,http://rnicrosoft.com,rnicrosoft.com,0,tidak_ditemukan_di_public_ti,Tidak Ada Temuan,,gagal,auth_key_belum_tersedia,Tidak ada temuan pada sumber threat intelligen...,Tidak ada temuan dari sumber threat intelligen...
5,http://paypal-verify-account.test,paypal-verify-account.test,0,tidak_ditemukan_di_public_ti,Tidak Ada Temuan,,gagal,auth_key_belum_tersedia,Tidak ada temuan pada sumber threat intelligen...,Tidak ada temuan dari sumber threat intelligen...
6,http://155.94.163.206/ai/?authenticated=true&a...,155.94.163.206,0,tidak_ditemukan_di_public_ti,Tidak Ada Temuan,,gagal,auth_key_belum_tersedia,Tidak ada temuan pada sumber threat intelligen...,Tidak ada temuan dari sumber threat intelligen...
7,https://www.google.com,www.google.com,0,tidak_ditemukan_di_public_ti,Tidak Ada Temuan,,gagal,auth_key_belum_tersedia,Tidak ada temuan pada sumber threat intelligen...,Tidak ada temuan dari sumber threat intelligen...


In [11]:
import phishrisk_engine_v4

importlib.reload(phishrisk_engine_v4)

engine_v4 = phishrisk_engine_v4.PhishRiskEngineV4(direktori_project)

hasil_engine_v4 = engine_v4.analisis_banyak_url(data_url_step13["url"].tolist())

lokasi_hasil_engine_v4 = direktori_outputs / "hasil_uji_engine_v4_step13.csv"
hasil_engine_v4.to_csv(lokasi_hasil_engine_v4, index=False)

print("Hasil uji Engine V4 disimpan:")
print(lokasi_hasil_engine_v4)

kolom_tampil = [
    "url",
    "hasil_akhir",
    "kategori_risiko",
    "skor_final",
    "public_ti_score",
    "public_ti_status",
    "public_ti_sources",
    "hasil_akhir_v4",
    "kategori_risiko_v4",
    "skor_final_v4",
    "rekomendasi_v4",
]
kolom_tampil = [kolom for kolom in kolom_tampil if kolom in hasil_engine_v4.columns]

display(hasil_engine_v4[kolom_tampil])

Hasil uji Engine V4 disimpan:
C:\Users\ASUS\PHISHING\reports\outputs\hasil_uji_engine_v4_step13.csv


,url,hasil_akhir,kategori_risiko,skor_final,public_ti_score,public_ti_status,public_ti_sources,hasil_akhir_v4,kategori_risiko_v4,skor_final_v4,rekomendasi_v4
0,https://praktikum.gunadarma.ac.id,Terlihat Aman,Rendah,20.40,0,tidak_ditemukan_di_public_ti,,Terlihat Aman,Rendah,20.40,Alamat cocok dengan daftar domain resmi dan ti...
1,https://www.bca.co.id,Terlihat Aman,Rendah,4.05,0,tidak_ditemukan_di_public_ti,,Terlihat Aman,Rendah,4.05,Alamat cocok dengan daftar domain resmi dan ti...
2,http://bca-login-update.test,Berisiko,Sangat Tinggi,99.60,0,tidak_ditemukan_di_public_ti,,Berisiko,Sangat Tinggi,99.60,Alamat terindikasi meniru brand atau domain re...
3,http://rricrosoft.com,Berisiko,Sangat Tinggi,99.60,0,tidak_ditemukan_di_public_ti,,Berisiko,Sangat Tinggi,99.60,Alamat terindikasi meniru brand atau domain re...
4,http://rnicrosoft.com,Berisiko,Sangat Tinggi,99.60,0,tidak_ditemukan_di_public_ti,,Berisiko,Sangat Tinggi,99.60,Alamat terindikasi meniru brand atau domain re...
5,http://paypal-verify-account.test,Berisiko,Sangat Tinggi,100.00,0,tidak_ditemukan_di_public_ti,,Berisiko,Sangat Tinggi,100.00,Alamat terindikasi meniru brand atau domain re...
6,http://155.94.163.206/ai/?authenticated=true&a...,Berisiko,Sangat Tinggi,100.00,0,tidak_ditemukan_di_public_ti,,Berisiko,Sangat Tinggi,100.00,"Alamat berisiko. Jangan dibuka, jangan diisi, ..."
7,https://www.google.com,Terlihat Aman,Rendah,24.00,0,tidak_ditemukan_di_public_ti,,Terlihat Aman,Rendah,24.00,Alamat cocok dengan daftar domain resmi dan ti...


In [12]:
lokasi_cli_output_step13 = direktori_outputs / "hasil_test_cli_engine_v4_step13.csv"

perintah_cli = [
    sys.executable,
    str(direktori_src / "run_phishrisk_v4.py"),
    "--mode",
    "urls",
    "--input",
    str(lokasi_input_step13),
    "--url-column",
    "url",
    "--output",
    str(lokasi_cli_output_step13),
]

hasil_cli = subprocess.run(
    perintah_cli,
    cwd=str(direktori_project),
    capture_output=True,
    text=True,
    env=os.environ.copy(),
)

print("STDOUT:")
print(hasil_cli.stdout)

print("STDERR:")
print(hasil_cli.stderr)

print("Return code:", hasil_cli.returncode)

if hasil_cli.returncode != 0:
    raise RuntimeError("CLI Engine V4 gagal dijalankan. Lihat STDERR di atas.")

print("Output CLI Engine V4:")
print(lokasi_cli_output_step13)

STDOUT:
Ringkasan Engine V4
hasil_akhir_v4
Berisiko         5
Terlihat Aman    3
Output: C:\Users\ASUS\PHISHING\reports\outputs\hasil_test_cli_engine_v4_step13.csv

STDERR:

Return code: 0
Output CLI Engine V4:
C:\Users\ASUS\PHISHING\reports\outputs\hasil_test_cli_engine_v4_step13.csv


In [13]:
lokasi_laporan_step13 = direktori_outputs / "laporan_step13_public_threat_intelligence.md"

ringkasan_v4 = hasil_engine_v4["hasil_akhir_v4"].value_counts().to_dict() if "hasil_akhir_v4" in hasil_engine_v4.columns else {}
ringkasan_public = hasil_public_ti["public_ti_status"].value_counts().to_dict() if "public_ti_status" in hasil_public_ti.columns else {}

isi_laporan_step13 = f'''# Laporan STEP 13 - Public Threat Intelligence

Waktu laporan: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}

## Tujuan

STEP 13 menambahkan sumber threat intelligence publik ke PhishRisk.

## API yang Ditambahkan

1. PhishTank
   - Fokus: URL phishing.
   - App key: opsional, tetapi tanpa key rate limit lebih terbatas.

2. URLhaus
   - Fokus: URL yang menyebarkan malware atau payload berbahaya.
   - Auth-Key: dibutuhkan untuk akses API modern abuse.ch.

## Output Utama

- `src/public_threat_intelligence.py`
- `src/phishrisk_engine_v4.py`
- `src/run_phishrisk_v4.py`
- `reports/outputs/hasil_uji_public_threat_intelligence_step13.csv`
- `reports/outputs/hasil_uji_engine_v4_step13.csv`

## Ringkasan Engine V4

{ringkasan_v4}

## Ringkasan Public Threat Intelligence

{ringkasan_public}

## Cara Membaca

- Jika Public Threat Intelligence menemukan URL di sumber eksternal, skor V4 dapat naik.
- Jika tidak ditemukan, hasil utama tetap mengikuti Engine V3.
- Jika API rate limit atau auth key belum tersedia, program tetap berjalan dengan status yang jelas.

## Catatan Keamanan

Public Threat Intelligence adalah sinyal tambahan. Jangan menjadikan satu API sebagai satu-satunya keputusan keamanan.
'''

lokasi_laporan_step13.write_text(isi_laporan_step13, encoding="utf-8")

print("Laporan STEP 13 disimpan:")
print(lokasi_laporan_step13)

print(isi_laporan_step13)

Laporan STEP 13 disimpan:
C:\Users\ASUS\PHISHING\reports\outputs\laporan_step13_public_threat_intelligence.md
# Laporan STEP 13 - Public Threat Intelligence

Waktu laporan: 2026-05-16 16:02:42

## Tujuan

STEP 13 menambahkan sumber threat intelligence publik ke PhishRisk.

## API yang Ditambahkan

1. PhishTank
   - Fokus: URL phishing.
   - App key: opsional, tetapi tanpa key rate limit lebih terbatas.

2. URLhaus
   - Fokus: URL yang menyebarkan malware atau payload berbahaya.
   - Auth-Key: dibutuhkan untuk akses API modern abuse.ch.

## Output Utama

- `src/public_threat_intelligence.py`
- `src/phishrisk_engine_v4.py`
- `src/run_phishrisk_v4.py`
- `reports/outputs/hasil_uji_public_threat_intelligence_step13.csv`
- `reports/outputs/hasil_uji_engine_v4_step13.csv`

## Ringkasan Engine V4

{'Berisiko': 5, 'Terlihat Aman': 3}

## Ringkasan Public Threat Intelligence

{'tidak_ditemukan_di_public_ti': 8}

## Cara Membaca

- Jika Public Threat Intelligence menemukan URL di sumber eksternal

In [14]:
lokasi_metadata_step13 = direktori_outputs / "metadata_step13_public_threat_intelligence.json"
lokasi_validasi_step13 = direktori_outputs / "validasi_step13_public_threat_intelligence.csv"

metadata_step13 = {
    "nama_program": "PhishRisk Public Threat Intelligence",
    "step": "STEP 13",
    "status": "selesai",
    "waktu": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "direktori_project": str(direktori_project),
    "api_ditambahkan": [
        {
            "nama": "PhishTank",
            "fungsi": "Pengecekan URL phishing",
            "auth": "Opsional app_key, rate limit lebih terbatas tanpa key",
            "env": "PHISHTANK_APP_KEY",
        },
        {
            "nama": "URLhaus",
            "fungsi": "Pengecekan URL malware/payload berbahaya",
            "auth": "Auth-Key gratis dari abuse.ch",
            "env": "URLHAUS_AUTH_KEY",
        },
    ],
    "file_modul_public_ti": str(direktori_src / "public_threat_intelligence.py"),
    "file_engine_v4": str(direktori_src / "phishrisk_engine_v4.py"),
    "file_cli_v4": str(direktori_src / "run_phishrisk_v4.py"),
    "input_uji": str(lokasi_input_step13),
    "output_public_ti": str(lokasi_hasil_public_ti),
    "output_engine_v4": str(lokasi_hasil_engine_v4),
    "output_cli": str(lokasi_cli_output_step13),
    "laporan": str(lokasi_laporan_step13),
    "catatan": "Public TI adalah sinyal tambahan. Engine V3 tetap menjadi dasar utama.",
}

lokasi_metadata_step13.write_text(
    json.dumps(metadata_step13, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

file_validasi_step13 = [
    direktori_src / "public_threat_intelligence.py",
    direktori_src / "phishrisk_engine_v4.py",
    direktori_src / "run_phishrisk_v4.py",
    lokasi_input_step13,
    lokasi_hasil_public_ti,
    lokasi_hasil_engine_v4,
    lokasi_cli_output_step13,
    lokasi_laporan_step13,
    lokasi_metadata_step13,
    direktori_project / ".env.example",
    direktori_project / ".gitignore",
]

data_validasi_step13 = []
for lokasi in file_validasi_step13:
    data_validasi_step13.append({
        "nama_file": lokasi.name,
        "lokasi": str(lokasi),
        "tersedia": lokasi.exists(),
        "ukuran_kb": round(lokasi.stat().st_size / 1024, 2) if lokasi.exists() else 0,
    })

data_validasi_step13 = pd.DataFrame(data_validasi_step13)
data_validasi_step13.to_csv(lokasi_validasi_step13, index=False)

print("Metadata STEP 13 disimpan:")
print(lokasi_metadata_step13)

print("\nValidasi STEP 13 disimpan:")
print(lokasi_validasi_step13)

display(data_validasi_step13)

if not data_validasi_step13["tersedia"].all():
    raise FileNotFoundError("Ada file hasil STEP 13 yang belum tersedia.")

print("\nRINGKASAN STEP 13")
print("=" * 70)
print("Modul Public Threat Intelligence:", direktori_src / "public_threat_intelligence.py")
print("Engine V4:", direktori_src / "phishrisk_engine_v4.py")
print("CLI V4:", direktori_src / "run_phishrisk_v4.py")
print("Output uji:", lokasi_hasil_engine_v4)
print("STEP 13 selesai.")

Metadata STEP 13 disimpan:
C:\Users\ASUS\PHISHING\reports\outputs\metadata_step13_public_threat_intelligence.json

Validasi STEP 13 disimpan:
C:\Users\ASUS\PHISHING\reports\outputs\validasi_step13_public_threat_intelligence.csv


,nama_file,lokasi,tersedia,ukuran_kb
0,public_threat_intelligence.py,C:\Users\ASUS\PHISHING\src\public_threat_intel...,True,13.04
1,phishrisk_engine_v4.py,C:\Users\ASUS\PHISHING\src\phishrisk_engine_v4.py,True,4.86
2,run_phishrisk_v4.py,C:\Users\ASUS\PHISHING\src\run_phishrisk_v4.py,True,3.22
3,input_url_step13_public_ti.csv,C:\Users\ASUS\PHISHING\examples\input_url_step...,True,0.25
4,hasil_uji_public_threat_intelligence_step13.csv,C:\Users\ASUS\PHISHING\reports\outputs\hasil_u...,True,4.02
5,hasil_uji_engine_v4_step13.csv,C:\Users\ASUS\PHISHING\reports\outputs\hasil_u...,True,8.38
6,hasil_test_cli_engine_v4_step13.csv,C:\Users\ASUS\PHISHING\reports\outputs\hasil_t...,True,8.38
7,laporan_step13_public_threat_intelligence.md,C:\Users\ASUS\PHISHING\reports\outputs\laporan...,True,1.23
8,metadata_step13_public_threat_intelligence.json,C:\Users\ASUS\PHISHING\reports\outputs\metadat...,True,1.42
9,.env.example,C:\Users\ASUS\PHISHING\.env.example,True,0.46



RINGKASAN STEP 13
Modul Public Threat Intelligence: C:\Users\ASUS\PHISHING\src\public_threat_intelligence.py
Engine V4: C:\Users\ASUS\PHISHING\src\phishrisk_engine_v4.py
CLI V4: C:\Users\ASUS\PHISHING\src\run_phishrisk_v4.py
Output uji: C:\Users\ASUS\PHISHING\reports\outputs\hasil_uji_engine_v4_step13.csv
STEP 13 selesai.


In [15]:
import pandas as pd

data_public_ti = pd.read_csv(
    r"C:\Users\ASUS\PHISHING\reports\outputs\hasil_uji_public_threat_intelligence_step13.csv"
)

kolom_cek = [
    "url",
    "phishtank_status",
    "phishtank_error",
    "urlhaus_query_status",
    "urlhaus_error",
]

kolom_cek = [kolom for kolom in kolom_cek if kolom in data_public_ti.columns]

display(data_public_ti[kolom_cek])

,url,phishtank_status,phishtank_error,urlhaus_query_status,urlhaus_error
0,https://praktikum.gunadarma.ac.id,gagal,Expecting value: line 1 column 1 (char 0),auth_key_belum_tersedia,URLhaus API membutuhkan Auth-Key gratis dari a...
1,https://www.bca.co.id,gagal,Expecting value: line 1 column 1 (char 0),auth_key_belum_tersedia,URLhaus API membutuhkan Auth-Key gratis dari a...
2,http://bca-login-update.test,gagal,Expecting value: line 1 column 1 (char 0),auth_key_belum_tersedia,URLhaus API membutuhkan Auth-Key gratis dari a...
3,http://rricrosoft.com,gagal,Expecting value: line 1 column 1 (char 0),auth_key_belum_tersedia,URLhaus API membutuhkan Auth-Key gratis dari a...
4,http://rnicrosoft.com,gagal,Expecting value: line 1 column 1 (char 0),auth_key_belum_tersedia,URLhaus API membutuhkan Auth-Key gratis dari a...
5,http://paypal-verify-account.test,gagal,Expecting value: line 1 column 1 (char 0),auth_key_belum_tersedia,URLhaus API membutuhkan Auth-Key gratis dari a...
6,http://155.94.163.206/ai/?authenticated=true&a...,gagal,Expecting value: line 1 column 1 (char 0),auth_key_belum_tersedia,URLhaus API membutuhkan Auth-Key gratis dari a...
7,https://www.google.com,gagal,Expecting value: line 1 column 1 (char 0),auth_key_belum_tersedia,URLhaus API membutuhkan Auth-Key gratis dari a...


In [16]:
from pathlib import Path

lokasi_public_ti = direktori_src / "public_threat_intelligence.py"

isi = lokasi_public_ti.read_text(encoding="utf-8")

isi = isi.replace(
    'endpoint = "http://checkurl.phishtank.com/checkurl/"',
    'endpoint = "https://checkurl.phishtank.com/checkurl/"'
)

isi = isi.replace(
'''        headers = {
            "User-Agent": "phishtank/phishrisk-intelligence-system",
        }''',
'''        headers = {
            "User-Agent": "phishtank/phishrisk-intelligence-system",
            "Accept": "application/json",
        }'''
)

isi = isi.replace(
'''            data = response.json()
            result = data.get("results", data if isinstance(data, dict) else {})''',
'''            teks_response = response.text.strip()

            if not teks_response:
                hasil["phishtank_status"] = "respons_kosong"
                hasil["phishtank_error"] = "PhishTank mengembalikan respons kosong."
                return hasil

            try:
                data = response.json()
            except Exception:
                hasil["phishtank_status"] = "respons_bukan_json"
                hasil["phishtank_error"] = _teks_pendek(
                    f"HTTP {response.status_code} | Content-Type: {response.headers.get('Content-Type', '')} | Preview: {teks_response[:300]}"
                )
                return hasil

            result = data.get("results", data if isinstance(data, dict) else {})'''
)

lokasi_public_ti.write_text(isi, encoding="utf-8")

print("Patch PhishTank selesai.")
print("Endpoint diganti ke HTTPS dan parsing JSON dibuat lebih aman.")
print(lokasi_public_ti)

Patch PhishTank selesai.
Endpoint diganti ke HTTPS dan parsing JSON dibuat lebih aman.
C:\Users\ASUS\PHISHING\src\public_threat_intelligence.py


In [17]:
import sys
import importlib
import pandas as pd

if str(direktori_src) not in sys.path:
    sys.path.insert(0, str(direktori_src))

import public_threat_intelligence
importlib.reload(public_threat_intelligence)

data_url_step13 = pd.read_csv(
    r"C:\Users\ASUS\PHISHING\examples\input_url_step13_public_ti.csv"
)

public_ti = public_threat_intelligence.PublicThreatIntelligence()

hasil_public_ti = public_ti.cek_banyak_url(data_url_step13["url"].tolist())

lokasi_hasil_public_ti = direktori_outputs / "hasil_uji_public_threat_intelligence_step13_patch.csv"
hasil_public_ti.to_csv(lokasi_hasil_public_ti, index=False)

kolom_cek = [
    "url",
    "public_ti_score",
    "public_ti_status",
    "phishtank_status",
    "phishtank_error",
    "urlhaus_query_status",
    "urlhaus_error",
]

kolom_cek = [kolom for kolom in kolom_cek if kolom in hasil_public_ti.columns]

display(hasil_public_ti[kolom_cek])

print("Hasil patch disimpan:")
print(lokasi_hasil_public_ti)

,url,public_ti_score,public_ti_status,phishtank_status,phishtank_error,urlhaus_query_status,urlhaus_error
0,https://praktikum.gunadarma.ac.id,0,tidak_ditemukan_di_public_ti,tidak_ditemukan,,auth_key_belum_tersedia,URLhaus API membutuhkan Auth-Key gratis dari a...
1,https://www.bca.co.id,0,tidak_ditemukan_di_public_ti,tidak_ditemukan,,auth_key_belum_tersedia,URLhaus API membutuhkan Auth-Key gratis dari a...
2,http://bca-login-update.test,0,tidak_ditemukan_di_public_ti,tidak_ditemukan,,auth_key_belum_tersedia,URLhaus API membutuhkan Auth-Key gratis dari a...
3,http://rricrosoft.com,0,tidak_ditemukan_di_public_ti,tidak_ditemukan,,auth_key_belum_tersedia,URLhaus API membutuhkan Auth-Key gratis dari a...
4,http://rnicrosoft.com,0,tidak_ditemukan_di_public_ti,tidak_ditemukan,,auth_key_belum_tersedia,URLhaus API membutuhkan Auth-Key gratis dari a...
5,http://paypal-verify-account.test,0,tidak_ditemukan_di_public_ti,tidak_ditemukan,,auth_key_belum_tersedia,URLhaus API membutuhkan Auth-Key gratis dari a...
6,http://155.94.163.206/ai/?authenticated=true&a...,0,tidak_ditemukan_di_public_ti,tidak_ditemukan,,auth_key_belum_tersedia,URLhaus API membutuhkan Auth-Key gratis dari a...
7,https://www.google.com,75,perlu_tinjauan_threat_intelligence,terdaftar_belum_valid,,auth_key_belum_tersedia,URLhaus API membutuhkan Auth-Key gratis dari a...


Hasil patch disimpan:
C:\Users\ASUS\PHISHING\reports\outputs\hasil_uji_public_threat_intelligence_step13_patch.csv


In [18]:
from pathlib import Path
import re

lokasi_public_ti = direktori_src / "public_threat_intelligence.py"

isi = lokasi_public_ti.read_text(encoding="utf-8")

pola = r'''    @staticmethod
    def hitung_skor_public_ti\(data: Dict\[str, Any\]\) -> Dict\[str, Any\]:
.*?
    @staticmethod
    def gabungkan_dengan_hasil_engine'''

fungsi_baru = r'''    @staticmethod
    def hitung_skor_public_ti(data: Dict[str, Any]) -> Dict[str, Any]:
        skor = 0
        sumber = []
        alasan = []

        phishtank_found = int(data.get("phishtank_found", 0) or 0)
        phishtank_verified = int(data.get("phishtank_verified", 0) or 0)
        phishtank_valid = int(data.get("phishtank_valid", 0) or 0)

        urlhaus_found = int(data.get("urlhaus_found", 0) or 0)

        if phishtank_found == 1:
            sumber.append("PhishTank")

            if phishtank_verified == 1 and phishtank_valid == 1:
                skor = max(skor, 100)
                alasan.append("URL terdaftar sebagai phishing aktif dan terverifikasi di PhishTank.")
            elif phishtank_verified == 1 and phishtank_valid == 0:
                skor = max(skor, 60)
                alasan.append("URL pernah muncul di PhishTank, tetapi statusnya tidak aktif atau belum valid saat ini.")
            else:
                skor = max(skor, 20)
                alasan.append("URL ditemukan di PhishTank, tetapi belum terverifikasi sebagai phishing aktif.")

        if urlhaus_found == 1:
            skor = max(skor, 100)
            sumber.append("URLhaus")
            alasan.append("URL ditemukan di URLhaus sebagai indikator malware atau payload berbahaya.")

        if skor >= 90:
            status = "terindikasi_ancaman_publik"
            kategori = "Sangat Tinggi"
            hasil = "Berisiko"
            rekomendasi = "URL ditemukan pada sumber threat intelligence publik. Jangan dibuka, jangan login, dan lakukan pengecekan manual."
        elif skor >= 60:
            status = "perlu_tinjauan_threat_intelligence"
            kategori = "Sedang"
            hasil = "Perlu Tinjauan"
            rekomendasi = "URL memiliki catatan pada sumber eksternal, tetapi belum cukup kuat untuk langsung dianggap berbahaya."
        elif skor > 0:
            status = "catatan_threat_intelligence_ringan"
            kategori = "Rendah"
            hasil = "Catatan Ringan"
            rekomendasi = "Ada catatan ringan dari sumber eksternal. Tetap gunakan hasil engine utama sebagai acuan."
        else:
            status = "tidak_ditemukan_di_public_ti"
            kategori = "Rendah"
            hasil = "Tidak Ada Temuan"
            rekomendasi = "Tidak ada temuan dari sumber threat intelligence publik yang aktif. Tetap gunakan hasil engine utama sebagai acuan."

        return {
            "public_ti_score": skor,
            "public_ti_status": status,
            "public_ti_category": kategori,
            "public_ti_result": hasil,
            "public_ti_sources": ", ".join(sorted(set(sumber))),
            "public_ti_reason": " ".join(alasan) if alasan else "Tidak ada temuan pada sumber threat intelligence publik yang aktif.",
            "public_ti_recommendation": rekomendasi,
        }

    @staticmethod
    def gabungkan_dengan_hasil_engine'''

isi_baru = re.sub(pola, fungsi_baru, isi, flags=re.DOTALL)

if isi == isi_baru:
    raise RuntimeError("Patch gagal diterapkan. Pola fungsi tidak ditemukan.")

lokasi_public_ti.write_text(isi_baru, encoding="utf-8")

print("Patch skor Public Threat Intelligence selesai.")
print("PhishTank belum valid sekarang tidak langsung menaikkan skor tinggi.")
print(lokasi_public_ti)

Patch skor Public Threat Intelligence selesai.
PhishTank belum valid sekarang tidak langsung menaikkan skor tinggi.
C:\Users\ASUS\PHISHING\src\public_threat_intelligence.py


In [19]:
import sys
import importlib
import pandas as pd

if str(direktori_src) not in sys.path:
    sys.path.insert(0, str(direktori_src))

import public_threat_intelligence
importlib.reload(public_threat_intelligence)

data_url_step13 = pd.read_csv(
    r"C:\Users\ASUS\PHISHING\examples\input_url_step13_public_ti.csv"
)

public_ti = public_threat_intelligence.PublicThreatIntelligence()

hasil_public_ti = public_ti.cek_banyak_url(data_url_step13["url"].tolist())

lokasi_hasil_public_ti_patch2 = direktori_outputs / "hasil_uji_public_threat_intelligence_step13_patch2.csv"
hasil_public_ti.to_csv(lokasi_hasil_public_ti_patch2, index=False)

kolom_cek = [
    "url",
    "public_ti_score",
    "public_ti_status",
    "public_ti_result",
    "phishtank_status",
    "phishtank_found",
    "phishtank_verified",
    "phishtank_valid",
    "urlhaus_query_status",
    "public_ti_reason",
]

kolom_cek = [kolom for kolom in kolom_cek if kolom in hasil_public_ti.columns]

display(hasil_public_ti[kolom_cek])

print("Hasil patch kedua disimpan:")
print(lokasi_hasil_public_ti_patch2)

,url,public_ti_score,public_ti_status,public_ti_result,phishtank_status,phishtank_found,phishtank_verified,phishtank_valid,urlhaus_query_status,public_ti_reason
0,https://praktikum.gunadarma.ac.id,0,tidak_ditemukan_di_public_ti,Tidak Ada Temuan,tidak_ditemukan,0,0,0,auth_key_belum_tersedia,Tidak ada temuan pada sumber threat intelligen...
1,https://www.bca.co.id,0,tidak_ditemukan_di_public_ti,Tidak Ada Temuan,tidak_ditemukan,0,0,0,auth_key_belum_tersedia,Tidak ada temuan pada sumber threat intelligen...
2,http://bca-login-update.test,0,tidak_ditemukan_di_public_ti,Tidak Ada Temuan,tidak_ditemukan,0,0,0,auth_key_belum_tersedia,Tidak ada temuan pada sumber threat intelligen...
3,http://rricrosoft.com,0,tidak_ditemukan_di_public_ti,Tidak Ada Temuan,tidak_ditemukan,0,0,0,auth_key_belum_tersedia,Tidak ada temuan pada sumber threat intelligen...
4,http://rnicrosoft.com,0,tidak_ditemukan_di_public_ti,Tidak Ada Temuan,tidak_ditemukan,0,0,0,auth_key_belum_tersedia,Tidak ada temuan pada sumber threat intelligen...
5,http://paypal-verify-account.test,0,tidak_ditemukan_di_public_ti,Tidak Ada Temuan,tidak_ditemukan,0,0,0,auth_key_belum_tersedia,Tidak ada temuan pada sumber threat intelligen...
6,http://155.94.163.206/ai/?authenticated=true&a...,0,tidak_ditemukan_di_public_ti,Tidak Ada Temuan,tidak_ditemukan,0,0,0,auth_key_belum_tersedia,Tidak ada temuan pada sumber threat intelligen...
7,https://www.google.com,60,perlu_tinjauan_threat_intelligence,Perlu Tinjauan,terdaftar_belum_valid,1,1,0,auth_key_belum_tersedia,"URL pernah muncul di PhishTank, tetapi statusn..."


Hasil patch kedua disimpan:
C:\Users\ASUS\PHISHING\reports\outputs\hasil_uji_public_threat_intelligence_step13_patch2.csv


In [20]:
import phishrisk_engine_v4
importlib.reload(phishrisk_engine_v4)

engine_v4 = phishrisk_engine_v4.PhishRiskEngineV4(direktori_project)

hasil_engine_v4_patch2 = engine_v4.analisis_banyak_url(data_url_step13["url"].tolist())

lokasi_hasil_engine_v4_patch2 = direktori_outputs / "hasil_uji_engine_v4_step13_patch2.csv"
hasil_engine_v4_patch2.to_csv(lokasi_hasil_engine_v4_patch2, index=False)

kolom_tampil = [
    "url",
    "hasil_akhir",
    "kategori_risiko",
    "skor_final",
    "public_ti_score",
    "public_ti_status",
    "public_ti_sources",
    "hasil_akhir_v4",
    "kategori_risiko_v4",
    "skor_final_v4",
    "rekomendasi_v4",
]

kolom_tampil = [kolom for kolom in kolom_tampil if kolom in hasil_engine_v4_patch2.columns]

display(hasil_engine_v4_patch2[kolom_tampil])

print("Hasil Engine V4 patch kedua disimpan:")
print(lokasi_hasil_engine_v4_patch2)

,url,hasil_akhir,kategori_risiko,skor_final,public_ti_score,public_ti_status,public_ti_sources,hasil_akhir_v4,kategori_risiko_v4,skor_final_v4,rekomendasi_v4
0,https://praktikum.gunadarma.ac.id,Terlihat Aman,Rendah,20.40,0,tidak_ditemukan_di_public_ti,,Terlihat Aman,Rendah,20.40,Alamat cocok dengan daftar domain resmi dan ti...
1,https://www.bca.co.id,Terlihat Aman,Rendah,4.05,0,tidak_ditemukan_di_public_ti,,Terlihat Aman,Rendah,4.05,Alamat cocok dengan daftar domain resmi dan ti...
2,http://bca-login-update.test,Berisiko,Sangat Tinggi,99.60,0,tidak_ditemukan_di_public_ti,,Berisiko,Sangat Tinggi,99.60,Alamat terindikasi meniru brand atau domain re...
3,http://rricrosoft.com,Berisiko,Sangat Tinggi,99.60,0,tidak_ditemukan_di_public_ti,,Berisiko,Sangat Tinggi,99.60,Alamat terindikasi meniru brand atau domain re...
4,http://rnicrosoft.com,Berisiko,Sangat Tinggi,99.60,0,tidak_ditemukan_di_public_ti,,Berisiko,Sangat Tinggi,99.60,Alamat terindikasi meniru brand atau domain re...
5,http://paypal-verify-account.test,Berisiko,Sangat Tinggi,100.00,0,tidak_ditemukan_di_public_ti,,Berisiko,Sangat Tinggi,100.00,Alamat terindikasi meniru brand atau domain re...
6,http://155.94.163.206/ai/?authenticated=true&a...,Berisiko,Sangat Tinggi,100.00,0,tidak_ditemukan_di_public_ti,,Berisiko,Sangat Tinggi,100.00,"Alamat berisiko. Jangan dibuka, jangan diisi, ..."
7,https://www.google.com,Terlihat Aman,Rendah,24.00,60,perlu_tinjauan_threat_intelligence,PhishTank,Terlihat Aman,Rendah,60.00,Alamat cocok dengan daftar domain resmi dan ti...


Hasil Engine V4 patch kedua disimpan:
C:\Users\ASUS\PHISHING\reports\outputs\hasil_uji_engine_v4_step13_patch2.csv


In [21]:
from pathlib import Path
import re

lokasi_public_ti = direktori_src / "public_threat_intelligence.py"

isi = lokasi_public_ti.read_text(encoding="utf-8")

pola = r'''    @staticmethod
    def hitung_skor_public_ti\(data: Dict\[str, Any\]\) -> Dict\[str, Any\]:
.*?
    @staticmethod
    def gabungkan_dengan_hasil_engine'''

fungsi_baru = r'''    @staticmethod
    def hitung_skor_public_ti(data: Dict[str, Any]) -> Dict[str, Any]:
        skor = 0
        sumber = []
        alasan = []

        phishtank_found = int(data.get("phishtank_found", 0) or 0)
        phishtank_verified = int(data.get("phishtank_verified", 0) or 0)
        phishtank_valid = int(data.get("phishtank_valid", 0) or 0)
        urlhaus_found = int(data.get("urlhaus_found", 0) or 0)

        if phishtank_found == 1:
            sumber.append("PhishTank")

            if phishtank_verified == 1 and phishtank_valid == 1:
                skor = max(skor, 100)
                alasan.append("URL terdaftar sebagai phishing aktif dan terverifikasi di PhishTank.")
            elif phishtank_verified == 1 and phishtank_valid == 0:
                skor = max(skor, 20)
                alasan.append("URL pernah muncul di PhishTank, tetapi statusnya tidak aktif atau belum valid saat ini.")
            else:
                skor = max(skor, 10)
                alasan.append("URL ditemukan di PhishTank, tetapi belum terverifikasi sebagai phishing aktif.")

        if urlhaus_found == 1:
            skor = max(skor, 100)
            sumber.append("URLhaus")
            alasan.append("URL ditemukan di URLhaus sebagai indikator malware atau payload berbahaya.")

        if skor >= 90:
            status = "terindikasi_ancaman_publik"
            kategori = "Sangat Tinggi"
            hasil = "Berisiko"
            rekomendasi = "URL ditemukan pada sumber threat intelligence publik. Jangan dibuka, jangan login, dan lakukan pengecekan manual."
        elif skor >= 60:
            status = "perlu_tinjauan_threat_intelligence"
            kategori = "Sedang"
            hasil = "Perlu Tinjauan"
            rekomendasi = "URL memiliki catatan kuat pada sumber eksternal, tetapi masih perlu validasi tambahan."
        elif skor > 0:
            status = "catatan_threat_intelligence_ringan"
            kategori = "Rendah"
            hasil = "Catatan Ringan"
            rekomendasi = "Ada catatan ringan dari sumber eksternal. Tetap gunakan hasil engine utama sebagai acuan."
        else:
            status = "tidak_ditemukan_di_public_ti"
            kategori = "Rendah"
            hasil = "Tidak Ada Temuan"
            rekomendasi = "Tidak ada temuan dari sumber threat intelligence publik yang aktif. Tetap gunakan hasil engine utama sebagai acuan."

        return {
            "public_ti_score": skor,
            "public_ti_status": status,
            "public_ti_category": kategori,
            "public_ti_result": hasil,
            "public_ti_sources": ", ".join(sorted(set(sumber))),
            "public_ti_reason": " ".join(alasan) if alasan else "Tidak ada temuan pada sumber threat intelligence publik yang aktif.",
            "public_ti_recommendation": rekomendasi,
        }

    @staticmethod
    def gabungkan_dengan_hasil_engine'''

isi_baru = re.sub(pola, fungsi_baru, isi, flags=re.DOTALL)

if isi == isi_baru:
    raise RuntimeError("Patch gagal diterapkan. Pola fungsi tidak ditemukan.")

lokasi_public_ti.write_text(isi_baru, encoding="utf-8")

print("Patch final Public Threat Intelligence selesai.")
print("PhishTank belum valid sekarang hanya menjadi catatan ringan.")
print(lokasi_public_ti)

Patch final Public Threat Intelligence selesai.
PhishTank belum valid sekarang hanya menjadi catatan ringan.
C:\Users\ASUS\PHISHING\src\public_threat_intelligence.py


In [22]:
import sys
import importlib
import pandas as pd

if str(direktori_src) not in sys.path:
    sys.path.insert(0, str(direktori_src))

import public_threat_intelligence
importlib.reload(public_threat_intelligence)

data_url_step13 = pd.read_csv(
    r"C:\Users\ASUS\PHISHING\examples\input_url_step13_public_ti.csv"
)

public_ti = public_threat_intelligence.PublicThreatIntelligence()

hasil_public_ti = public_ti.cek_banyak_url(data_url_step13["url"].tolist())

lokasi_hasil_public_ti_patch3 = direktori_outputs / "hasil_uji_public_threat_intelligence_step13_patch3.csv"
hasil_public_ti.to_csv(lokasi_hasil_public_ti_patch3, index=False)

kolom_cek = [
    "url",
    "public_ti_score",
    "public_ti_status",
    "public_ti_result",
    "phishtank_status",
    "phishtank_found",
    "phishtank_verified",
    "phishtank_valid",
    "urlhaus_query_status",
    "public_ti_reason",
]

kolom_cek = [kolom for kolom in kolom_cek if kolom in hasil_public_ti.columns]

display(hasil_public_ti[kolom_cek])

print("Hasil patch final disimpan:")
print(lokasi_hasil_public_ti_patch3)

,url,public_ti_score,public_ti_status,public_ti_result,phishtank_status,phishtank_found,phishtank_verified,phishtank_valid,urlhaus_query_status,public_ti_reason
0,https://praktikum.gunadarma.ac.id,0,tidak_ditemukan_di_public_ti,Tidak Ada Temuan,tidak_ditemukan,0,0,0,auth_key_belum_tersedia,Tidak ada temuan pada sumber threat intelligen...
1,https://www.bca.co.id,0,tidak_ditemukan_di_public_ti,Tidak Ada Temuan,tidak_ditemukan,0,0,0,auth_key_belum_tersedia,Tidak ada temuan pada sumber threat intelligen...
2,http://bca-login-update.test,0,tidak_ditemukan_di_public_ti,Tidak Ada Temuan,tidak_ditemukan,0,0,0,auth_key_belum_tersedia,Tidak ada temuan pada sumber threat intelligen...
3,http://rricrosoft.com,0,tidak_ditemukan_di_public_ti,Tidak Ada Temuan,tidak_ditemukan,0,0,0,auth_key_belum_tersedia,Tidak ada temuan pada sumber threat intelligen...
4,http://rnicrosoft.com,0,tidak_ditemukan_di_public_ti,Tidak Ada Temuan,tidak_ditemukan,0,0,0,auth_key_belum_tersedia,Tidak ada temuan pada sumber threat intelligen...
5,http://paypal-verify-account.test,0,tidak_ditemukan_di_public_ti,Tidak Ada Temuan,tidak_ditemukan,0,0,0,auth_key_belum_tersedia,Tidak ada temuan pada sumber threat intelligen...
6,http://155.94.163.206/ai/?authenticated=true&a...,0,tidak_ditemukan_di_public_ti,Tidak Ada Temuan,tidak_ditemukan,0,0,0,auth_key_belum_tersedia,Tidak ada temuan pada sumber threat intelligen...
7,https://www.google.com,20,catatan_threat_intelligence_ringan,Catatan Ringan,terdaftar_belum_valid,1,1,0,auth_key_belum_tersedia,"URL pernah muncul di PhishTank, tetapi statusn..."


Hasil patch final disimpan:
C:\Users\ASUS\PHISHING\reports\outputs\hasil_uji_public_threat_intelligence_step13_patch3.csv


In [23]:
import phishrisk_engine_v4
importlib.reload(phishrisk_engine_v4)

engine_v4 = phishrisk_engine_v4.PhishRiskEngineV4(direktori_project)

hasil_engine_v4_patch3 = engine_v4.analisis_banyak_url(data_url_step13["url"].tolist())

lokasi_hasil_engine_v4_patch3 = direktori_outputs / "hasil_uji_engine_v4_step13_patch3.csv"
hasil_engine_v4_patch3.to_csv(lokasi_hasil_engine_v4_patch3, index=False)

kolom_tampil = [
    "url",
    "hasil_akhir",
    "kategori_risiko",
    "skor_final",
    "public_ti_score",
    "public_ti_status",
    "public_ti_sources",
    "hasil_akhir_v4",
    "kategori_risiko_v4",
    "skor_final_v4",
    "rekomendasi_v4",
]

kolom_tampil = [kolom for kolom in kolom_tampil if kolom in hasil_engine_v4_patch3.columns]

display(hasil_engine_v4_patch3[kolom_tampil])

print("Hasil Engine V4 patch final disimpan:")
print(lokasi_hasil_engine_v4_patch3)

,url,hasil_akhir,kategori_risiko,skor_final,public_ti_score,public_ti_status,public_ti_sources,hasil_akhir_v4,kategori_risiko_v4,skor_final_v4,rekomendasi_v4
0,https://praktikum.gunadarma.ac.id,Terlihat Aman,Rendah,20.40,0,tidak_ditemukan_di_public_ti,,Terlihat Aman,Rendah,20.40,Alamat cocok dengan daftar domain resmi dan ti...
1,https://www.bca.co.id,Terlihat Aman,Rendah,4.05,0,tidak_ditemukan_di_public_ti,,Terlihat Aman,Rendah,4.05,Alamat cocok dengan daftar domain resmi dan ti...
2,http://bca-login-update.test,Berisiko,Sangat Tinggi,99.60,0,tidak_ditemukan_di_public_ti,,Berisiko,Sangat Tinggi,99.60,Alamat terindikasi meniru brand atau domain re...
3,http://rricrosoft.com,Berisiko,Sangat Tinggi,99.60,0,tidak_ditemukan_di_public_ti,,Berisiko,Sangat Tinggi,99.60,Alamat terindikasi meniru brand atau domain re...
4,http://rnicrosoft.com,Berisiko,Sangat Tinggi,99.60,0,tidak_ditemukan_di_public_ti,,Berisiko,Sangat Tinggi,99.60,Alamat terindikasi meniru brand atau domain re...
5,http://paypal-verify-account.test,Berisiko,Sangat Tinggi,100.00,0,tidak_ditemukan_di_public_ti,,Berisiko,Sangat Tinggi,100.00,Alamat terindikasi meniru brand atau domain re...
6,http://155.94.163.206/ai/?authenticated=true&a...,Berisiko,Sangat Tinggi,100.00,0,tidak_ditemukan_di_public_ti,,Berisiko,Sangat Tinggi,100.00,"Alamat berisiko. Jangan dibuka, jangan diisi, ..."
7,https://www.google.com,Terlihat Aman,Rendah,24.00,20,catatan_threat_intelligence_ringan,PhishTank,Terlihat Aman,Rendah,24.00,Alamat cocok dengan daftar domain resmi dan ti...


Hasil Engine V4 patch final disimpan:
C:\Users\ASUS\PHISHING\reports\outputs\hasil_uji_engine_v4_step13_patch3.csv


In [24]:
from pathlib import Path
import json
import pandas as pd
from datetime import datetime

lokasi_public_ti_patch3 = direktori_outputs / "hasil_uji_public_threat_intelligence_step13_patch3.csv"
lokasi_engine_v4_patch3 = direktori_outputs / "hasil_uji_engine_v4_step13_patch3.csv"

lokasi_public_ti_final = direktori_outputs / "hasil_uji_public_threat_intelligence_step13.csv"
lokasi_engine_v4_final = direktori_outputs / "hasil_uji_engine_v4_step13.csv"
lokasi_laporan_final = direktori_outputs / "laporan_step13_public_threat_intelligence.md"
lokasi_metadata_final = direktori_outputs / "metadata_step13_public_threat_intelligence.json"
lokasi_validasi_final = direktori_outputs / "validasi_step13_public_threat_intelligence.csv"

data_public_ti_final = pd.read_csv(lokasi_public_ti_patch3)
data_engine_v4_final = pd.read_csv(lokasi_engine_v4_patch3)

data_public_ti_final.to_csv(lokasi_public_ti_final, index=False)
data_engine_v4_final.to_csv(lokasi_engine_v4_final, index=False)

ringkasan_engine_v4 = data_engine_v4_final["hasil_akhir_v4"].value_counts().to_dict()
ringkasan_public_ti = data_public_ti_final["public_ti_status"].value_counts().to_dict()

isi_laporan_final = f"""# Laporan STEP 13 - Public Threat Intelligence

Waktu laporan: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}

## Tujuan

STEP 13 menambahkan Public Threat Intelligence ke PhishRisk agar hasil deteksi tidak hanya bergantung pada model lokal.

## API yang Digunakan

1. PhishTank  
   Digunakan untuk melihat apakah URL pernah tercatat sebagai phishing.

2. URLhaus  
   Digunakan untuk melihat apakah URL pernah tercatat sebagai URL malware atau payload berbahaya.

## Hasil Final

Ringkasan Engine V4:

{ringkasan_engine_v4}

Ringkasan Public Threat Intelligence:

{ringkasan_public_ti}

## Catatan Penting

- PhishTank sudah berhasil dibaca.
- URL yang ditemukan tetapi belum valid hanya menjadi catatan ringan.
- URLhaus belum aktif penuh karena membutuhkan Auth-Key.
- Engine V4 tetap memakai Engine V3 sebagai dasar utama.
- Public Threat Intelligence hanya menjadi sinyal tambahan.

## Kesimpulan

STEP 13 berhasil. Program sekarang memiliki lapisan tambahan Public Threat Intelligence tanpa merusak hasil utama dari Engine V3.
"""

lokasi_laporan_final.write_text(isi_laporan_final, encoding="utf-8")

metadata_final = {
    "nama_program": "PhishRisk Public Threat Intelligence",
    "step": "STEP 13",
    "status": "selesai dan sudah dikalibrasi",
    "waktu": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "komponen_baru": [
        "public_threat_intelligence.py",
        "phishrisk_engine_v4.py",
        "run_phishrisk_v4.py"
    ],
    "api_digunakan": [
        "PhishTank",
        "URLhaus"
    ],
    "catatan_phishtank": "PhishTank berhasil dibaca. Status belum valid hanya dihitung sebagai catatan ringan.",
    "catatan_urlhaus": "URLhaus membutuhkan Auth-Key agar aktif penuh.",
    "output_public_ti": str(lokasi_public_ti_final),
    "output_engine_v4": str(lokasi_engine_v4_final),
    "laporan": str(lokasi_laporan_final)
}

lokasi_metadata_final.write_text(
    json.dumps(metadata_final, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

file_validasi = [
    direktori_src / "public_threat_intelligence.py",
    direktori_src / "phishrisk_engine_v4.py",
    direktori_src / "run_phishrisk_v4.py",
    lokasi_public_ti_final,
    lokasi_engine_v4_final,
    lokasi_laporan_final,
    lokasi_metadata_final,
    direktori_project / ".env.example",
    direktori_project / ".gitignore",
]

data_validasi_final = []

for lokasi in file_validasi:
    data_validasi_final.append({
        "nama_file": lokasi.name,
        "lokasi": str(lokasi),
        "tersedia": lokasi.exists(),
        "ukuran_kb": round(lokasi.stat().st_size / 1024, 2) if lokasi.exists() else 0
    })

data_validasi_final = pd.DataFrame(data_validasi_final)
data_validasi_final.to_csv(lokasi_validasi_final, index=False)

display(data_validasi_final)

print("Finalisasi STEP 13 selesai.")
print("Output Public TI final:", lokasi_public_ti_final)
print("Output Engine V4 final:", lokasi_engine_v4_final)
print("Laporan final:", lokasi_laporan_final)
print("Metadata final:", lokasi_metadata_final)
print("Validasi final:", lokasi_validasi_final)

,nama_file,lokasi,tersedia,ukuran_kb
0,public_threat_intelligence.py,C:\Users\ASUS\PHISHING\src\public_threat_intel...,True,14.30
1,phishrisk_engine_v4.py,C:\Users\ASUS\PHISHING\src\phishrisk_engine_v4.py,True,4.86
2,run_phishrisk_v4.py,C:\Users\ASUS\PHISHING\src\run_phishrisk_v4.py,True,3.22
3,hasil_uji_public_threat_intelligence_step13.csv,C:\Users\ASUS\PHISHING\reports\outputs\hasil_u...,True,3.85
4,hasil_uji_engine_v4_step13.csv,C:\Users\ASUS\PHISHING\reports\outputs\hasil_u...,True,8.21
5,laporan_step13_public_threat_intelligence.md,C:\Users\ASUS\PHISHING\reports\outputs\laporan...,True,1.09
6,metadata_step13_public_threat_intelligence.json,C:\Users\ASUS\PHISHING\reports\outputs\metadat...,True,0.84
7,.env.example,C:\Users\ASUS\PHISHING\.env.example,True,0.46
8,.gitignore,C:\Users\ASUS\PHISHING\.gitignore,True,0.44


Finalisasi STEP 13 selesai.
Output Public TI final: C:\Users\ASUS\PHISHING\reports\outputs\hasil_uji_public_threat_intelligence_step13.csv
Output Engine V4 final: C:\Users\ASUS\PHISHING\reports\outputs\hasil_uji_engine_v4_step13.csv
Laporan final: C:\Users\ASUS\PHISHING\reports\outputs\laporan_step13_public_threat_intelligence.md
Metadata final: C:\Users\ASUS\PHISHING\reports\outputs\metadata_step13_public_threat_intelligence.json
Validasi final: C:\Users\ASUS\PHISHING\reports\outputs\validasi_step13_public_threat_intelligence.csv
